# MedBIND3D v16 — Train 3D U-Net with Modality Dropout from Scratch

**The only reliable fix for missing-modality TC/ET recovery:**
Train a model that has actually SEEN missing modalities during training.

**Architecture:** Lightweight 3D U-Net (fits RTX 4060 8GB)
**Key:** Random modality dropout during training — model learns to segment from any subset of modalities
**Expected training time:** ~12-16 hours on RTX 4060
**Expected results:** ET missing T1CE ~0.55-0.65 (vs current 0.59 with nnU-Net zero-input)

After training, wrap our existing Memory-TTA + Mamba2 adapter on top.

**All outputs -> `./medbind3d_v16_outputs/` with `v16_` prefix**

In [1]:
# ── CELL 1: Setup ──────────────────────────────────────────────────────────────
import numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F, nibabel as nib, cv2
from pathlib import Path
from tqdm import tqdm
from scipy.stats import ttest_rel
from scipy.ndimage import zoom
import os, warnings, shutil, random, traceback, gc
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

os.chdir('C:/Users/arnav/Desktop/MedBIND3D/MedBIND3D/medclipsam/MedCLIP-SAMv2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_ROOT    = 'C:/Users/arnav/Desktop/MedBIND3D/MedBIND3D/BraTS2020_training_data/MICCAI_BraTS2020_TrainingData'
OUTPUT_DIR   = './medbind3d_v16_outputs'
MODEL_CKPT   = f'{OUTPUT_DIR}/v16_unet3d_modality_dropout.pth'
CACHE_DIR    = f'{OUTPUT_DIR}/pred_cache'

# patch size — fits in 8GB VRAM
PATCH_H, PATCH_W, PATCH_D = 96, 96, 96

MODS    = ['FLAIR','T1','T1CE','T2']
REGIONS = ['WT','TC','ET']
N_TEST  = 20

for d in [OUTPUT_DIR, CACHE_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Device: {device}')
print(f'Outputs -> {os.path.abspath(OUTPUT_DIR)}')
print('Setup complete')

Device: cuda
Outputs -> C:\Users\arnav\Desktop\MedBIND3D\MedBIND3D\medclipsam\MedCLIP-SAMv2\medbind3d_v16_outputs
Setup complete


In [2]:
# ── CELL 2: Dataset ────────────────────────────────────────────────────────────

def get_all_patient_dirs(root):
    return [d for d in sorted(Path(root).iterdir())
            if d.is_dir() and all(len(list(d.glob(f'*{s}*')))>0
            for s in ['t1.nii','t1ce.nii','t2.nii','flair.nii','seg.nii'])]

def load_patient(patient_dir):
    pid = patient_dir.name; vols = []
    for m in ['flair','t1','t1ce','t2']:
        f = list(patient_dir.glob(f'*{m}.nii*'))[0]
        v = nib.as_closest_canonical(nib.load(str(f))).get_fdata(dtype=np.float32)
        mask = v > 0
        if mask.sum() > 0:
            v[mask] = (v[mask]-v[mask].mean())/(v[mask].std()+1e-8)
        vols.append(v)
    seg_f = list(patient_dir.glob('*seg.nii*'))[0]
    seg   = nib.as_closest_canonical(nib.load(str(seg_f))).get_fdata().astype(np.int16)
    return np.stack(vols, axis=0), seg, pid  # [4,H,W,D], [H,W,D]

def gt_regions(seg):
    return {'WT':(seg>0).astype(np.float32),
            'TC':((seg==1)|(seg==4)).astype(np.float32),
            'ET':(seg==4).astype(np.float32)}

def dice_3d(pred, gt):
    i=np.sum(pred*gt); u=np.sum(pred)+np.sum(gt)
    return 2.0*i/u if u>0 else 0.0

all_dirs  = get_all_patient_dirs(DATA_ROOT)
test_dirs = all_dirs[:N_TEST]
train_dirs= all_dirs[N_TEST:]
print(f'Train: {len(train_dirs)}  Test: {len(test_dirs)}')

Train: 348  Test: 20


In [3]:
# ── CELL 3: Lightweight 3D U-Net Architecture ────────────────────────────────

class ConvBlock3D(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class UNet3DModalityDropout(nn.Module):
    """
    Lightweight 3D U-Net trained with modality dropout.
    Input: [B, 4, H, W, D] — any modality can be zeroed at inference.
    Output: [B, 4, H, W, D] — class logits (BG, NCR, ED, ET)
    Filters: 16→32→64→128 (fits 8GB VRAM at 96^3 patches)
    """
    def __init__(self, in_ch=4, num_classes=4, base=16):
        super().__init__()
        self.enc1 = ConvBlock3D(in_ch, base)
        self.enc2 = ConvBlock3D(base, base*2)
        self.enc3 = ConvBlock3D(base*2, base*4)
        self.enc4 = ConvBlock3D(base*4, base*8)
        self.pool = nn.MaxPool3d(2)
        self.up3  = nn.ConvTranspose3d(base*8, base*4, 2, stride=2)
        self.dec3 = ConvBlock3D(base*8, base*4)
        self.up2  = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.dec2 = ConvBlock3D(base*4, base*2)
        self.up1  = nn.ConvTranspose3d(base*2, base, 2, stride=2)
        self.dec1 = ConvBlock3D(base*2, base)
        self.out  = nn.Conv3d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(e4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)

model = UNet3DModalityDropout(in_ch=4, num_classes=4, base=16).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'3D U-Net: {n_params/1e6:.2f}M params')

# VRAM check
_x = torch.randn(1, 4, PATCH_H, PATCH_W, PATCH_D).to(device)
_o = model(_x)
assert _o.shape == (1, 4, PATCH_H, PATCH_W, PATCH_D)
print(f'SMOKE TEST PASSED: output shape {_o.shape}')
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'VRAM: {used:.2f}GB used / {total:.2f}GB total')
del _x, _o
gc.collect(); torch.cuda.empty_cache()

3D U-Net: 1.40M params
SMOKE TEST PASSED: output shape torch.Size([1, 4, 96, 96, 96])
VRAM: 0.81GB used / 8.59GB total


In [4]:
# ── CELL 4: Training Functions ─────────────────────────────────────────────────

def apply_modality_dropout(x, dropout_prob=0.5):
    """
    Randomly zero out 1-2 modality channels during training.
    This is the key to missing-modality robustness.
    dropout_prob: probability of dropping any individual modality
    """
    mask = (torch.rand(x.shape[0], x.shape[1], 1, 1, 1, device=x.device) > dropout_prob).float()
    # Always keep at least 2 modalities (don't zero all channels)
    n_kept = mask.sum(dim=1, keepdim=True)
    # If fewer than 2 kept, restore random channels
    for b in range(x.shape[0]):
        if mask[b].sum() < 2:
            restore = torch.randperm(4)[:2]
            mask[b, restore] = 1.0
    return x * mask

def random_3d_crop(vol4ch, seg, ph=PATCH_H, pw=PATCH_W, pd_sz=PATCH_D):
    """Random patch crop with foreground/tumor bias."""
    H, W, D = seg.shape
    # 70% tumor-centered, 30% random
    if seg.sum() > 0 and random.random() < 0.7:
        tumor_voxels = np.argwhere(seg > 0)
        center = tumor_voxels[np.random.randint(len(tumor_voxels))]
        h0 = np.clip(center[0]-ph//2, 0, max(0, H-ph))
        w0 = np.clip(center[1]-pw//2, 0, max(0, W-pw))
        d0 = np.clip(center[2]-pd_sz//2, 0, max(0, D-pd_sz))
    else:
        h0 = random.randint(0, max(0, H-ph))
        w0 = random.randint(0, max(0, W-pw))
        d0 = random.randint(0, max(0, D-pd_sz))
    patch_v = vol4ch[:, h0:h0+ph, w0:w0+pw, d0:d0+pd_sz]
    patch_s = seg[h0:h0+ph, w0:w0+pw, d0:d0+pd_sz]
    # Pad if needed (edge of volume)
    if patch_s.shape != (ph, pw, pd_sz):
        pad_v = [(0,0)] + [(0, ph-patch_v.shape[1]), (0, pw-patch_v.shape[2]), (0, pd_sz-patch_v.shape[3])]
        patch_v = np.pad(patch_v, pad_v); patch_s = np.pad(patch_s, pad_v[1:])
    return patch_v, patch_s

def seg_to_onehot(seg_np, n_classes=4):
    """BraTS labels: 0=BG, 1=NCR, 2=ED, 4=ET → remap 4→3."""
    s = seg_np.copy(); s[s==4] = 3
    onehot = np.zeros((n_classes,)+s.shape, dtype=np.float32)
    for c in range(n_classes): onehot[c] = (s==c)
    return onehot

def soft_dice_loss_multiclass(logits, target_onehot, weights=[0.5, 1.0, 1.0, 2.0]):
    """Weighted multiclass Dice loss — ET gets 2× weight."""
    probs = torch.softmax(logits, dim=1)
    loss = 0.0
    for c, w in enumerate(weights):
        p = probs[:,c].reshape(-1); t = target_onehot[:,c].reshape(-1)
        i=(p*t).sum(); u=p.sum()+t.sum()
        loss += w*(1-(2*i+1)/(u+1))
    return loss / sum(weights)

def random_flip_3d(vol, seg):
    """Random flipping augmentation."""
    axes = [ax for ax in [2,3,4] if random.random()>0.5]
    for ax in axes:
        vol = torch.flip(vol, [ax]); seg = torch.flip(seg, [ax-1])
    return vol, seg

print('Training functions ready')

Training functions ready


In [5]:
# ── CELL 5: TRAIN (the expensive step — ~12-16 hrs on RTX 4060) ───────────────
N_EPOCHS           = 50
PATCHES_PER_PATIENT= 4     # patches sampled per patient per epoch
LR                 = 1e-3
MODALITY_DROPOUT_PROB = 0.5  # prob of dropping each modality

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS)

# SMOKE TEST before 12 hrs
print('Running training smoke test (1 patient, no try/except)...')
_vols, _seg, _pid = load_patient(train_dirs[0])
_patch_v, _patch_s = random_3d_crop(_vols, _seg)
_x = torch.FloatTensor(_patch_v).unsqueeze(0).to(device)
_x_drop = apply_modality_dropout(_x)
_y = torch.FloatTensor(seg_to_onehot(_patch_s)).unsqueeze(0).to(device)
_logits = model(_x_drop)
_loss = soft_dice_loss_multiclass(_logits, _y)
_loss.backward()
optimizer.zero_grad()
print(f'SMOKE TEST PASSED: loss={_loss.item():.4f}  input_shape={_x.shape}')
del _vols,_seg,_patch_v,_patch_s,_x,_x_drop,_y,_logits,_loss
gc.collect(); torch.cuda.empty_cache()

print(f'\nTraining {N_EPOCHS} epochs on {len(train_dirs)} patients')
print(f'{PATCHES_PER_PATIENT} patches/patient/epoch = {len(train_dirs)*PATCHES_PER_PATIENT} steps/epoch')
print(f'Modality dropout prob: {MODALITY_DROPOUT_PROB} per channel')
print(f'Expected time: ~{N_EPOCHS*len(train_dirs)*PATCHES_PER_PATIENT*1.5/3600:.1f} hours\n')

train_losses = []
for epoch in range(N_EPOCHS):
    model.train()
    ep_loss = 0.0; n_steps = 0; n_failed = 0
    random.shuffle(train_dirs)
    for pd_ in tqdm(train_dirs, desc=f'Epoch {epoch+1}/{N_EPOCHS}', leave=False):
        try:
            vols, seg, _ = load_patient(pd_)
            for _ in range(PATCHES_PER_PATIENT):
                patch_v, patch_s = random_3d_crop(vols, seg)
                x = torch.FloatTensor(patch_v).unsqueeze(0).to(device)
                y = torch.FloatTensor(seg_to_onehot(patch_s)).unsqueeze(0).to(device)
                # Modality dropout: 50% of patches drop 1-2 modalities
                if random.random() < 0.5:
                    x = apply_modality_dropout(x, dropout_prob=MODALITY_DROPOUT_PROB)
                # Random flip augmentation
                x, y_seg = random_flip_3d(x, torch.FloatTensor(patch_s).unsqueeze(0).to(device))
                optimizer.zero_grad()
                logits = model(x)
                loss = soft_dice_loss_multiclass(logits, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                ep_loss += loss.item(); n_steps += 1
            del vols, seg, patch_v, patch_s, x, y
        except Exception as e:
            n_failed += 1; print(f'  [FAILED] {pd_.name}: {e}'); traceback.print_exc(); continue
    gc.collect(); torch.cuda.empty_cache()
    if n_steps == 0: raise RuntimeError(f'Epoch {epoch+1}: zero steps.')
    scheduler.step()
    avg = ep_loss/n_steps; train_losses.append(avg)
    print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  Loss: {avg:.4f}  LR: {scheduler.get_last_lr()[0]:.6f}  (failed={n_failed})')
    # Save checkpoint every 10 epochs
    if (epoch+1) % 10 == 0:
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/v16_epoch{epoch+1}.pth')
        print(f'  Checkpoint saved: epoch {epoch+1}')

torch.save(model.state_dict(), MODEL_CKPT)
print(f'\nFinal model saved -> {MODEL_CKPT}')
plt.figure(figsize=(7,3))
plt.plot(train_losses); plt.grid(alpha=0.3); plt.xlabel('Epoch'); plt.ylabel('Dice loss')
plt.title('3D U-Net with modality dropout — training loss')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/v16_training_curve.png', dpi=300, bbox_inches='tight')
plt.close(); print('Training curve saved')

Running training smoke test (1 patient, no try/except)...
SMOKE TEST PASSED: loss=0.9362  input_shape=torch.Size([1, 4, 96, 96, 96])

Training 50 epochs on 348 patients
4 patches/patient/epoch = 1392 steps/epoch
Modality dropout prob: 0.5 per channel
Expected time: ~29.0 hours



Epoch   1/50  Loss: 0.8363  LR: 0.000999  (failed=0)


Epoch   2/50  Loss: 0.7996  LR: 0.000996  (failed=0)


Epoch   3/50  Loss: 0.7880  LR: 0.000991  (failed=0)


Epoch   4/50  Loss: 0.7781  LR: 0.000984  (failed=0)


...

Epoch  46/50  Loss: 0.6675  LR: 0.000016  (failed=0)


Epoch  47/50  Loss: 0.6891  LR: 0.000009  (failed=0)


Epoch  48/50  Loss: 0.6812  LR: 0.000004  (failed=0)


Epoch  49/50  Loss: 0.6733  LR: 0.000001  (failed=0)


Epoch  50/50  Loss: 0.6730  LR: 0.000000  (failed=0)
  Checkpoint saved: epoch 50

Final model saved -> ./medbind3d_v16_outputs/v16_unet3d_modality_dropout.pth
Training curve saved


In [6]:
# ── CELL 6: Sliding Window Inference ──────────────────────────────────────────
# Load trained model
model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
model.eval()
print('Model loaded for inference')

def sliding_window_inference(model, vol4ch, patch_h=PATCH_H, patch_w=PATCH_W,
                              patch_d=PATCH_D, stride=48):
    """Sliding window inference over full 3D volume with overlap averaging."""
    H, W, D = vol4ch.shape[1], vol4ch.shape[2], vol4ch.shape[3]
    probs_sum = np.zeros((4, H, W, D), dtype=np.float32)
    count     = np.zeros((1, H, W, D), dtype=np.float32)

    h_steps = list(range(0, max(1,H-patch_h+1), stride)) + [max(0,H-patch_h)]
    w_steps = list(range(0, max(1,W-patch_w+1), stride)) + [max(0,W-patch_w)]
    d_steps = list(range(0, max(1,D-patch_d+1), stride)) + [max(0,D-patch_d)]
    h_steps = sorted(set(h_steps)); w_steps = sorted(set(w_steps)); d_steps = sorted(set(d_steps))

    model.eval()
    with torch.no_grad():
        for h0 in h_steps:
            for w0 in w_steps:
                for d0 in d_steps:
                    h1=min(h0+patch_h,H); w1=min(w0+patch_w,W); d1=min(d0+patch_d,D)
                    patch = vol4ch[:, h0:h1, w0:w1, d0:d1]
                    # Pad if needed
                    ph, pw, pd_sz = h1-h0, w1-w0, d1-d0
                    if ph<patch_h or pw<patch_w or pd_sz<patch_d:
                        pad = [(0,0),(0,patch_h-ph),(0,patch_w-pw),(0,patch_d-pd_sz)]
                        patch = np.pad(patch, pad)
                    x = torch.FloatTensor(patch).unsqueeze(0).to(device)
                    logits = model(x)
                    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
                    probs_sum[:, h0:h1, w0:w1, d0:d1] += probs[:, :ph, :pw, :pd_sz]
                    count[0, h0:h1, w0:w1, d0:d1] += 1

    probs_avg = probs_sum / (count + 1e-8)
    # Map back to WT/TC/ET: class 0=BG, 1=NCR, 2=ED, 3=ET(was 4)
    return {
        'WT': (probs_avg[1]+probs_avg[2]+probs_avg[3] > 0.5).astype(float),
        'TC': (probs_avg[1]+probs_avg[3] > 0.5).astype(float),
        'ET': (probs_avg[3] > 0.5).astype(float),
    }

# SMOKE TEST
print('Running inference smoke test...')
_vols, _seg, _pid = load_patient(test_dirs[0])
_gtr = gt_regions(_seg)
_preds = sliding_window_inference(model, _vols)
for r in REGIONS:
    print(f'  {r}: {dice_3d(_preds[r], _gtr[r]):.3f}')
del _vols, _seg, _gtr, _preds
gc.collect(); torch.cuda.empty_cache()
print('Smoke test passed')

Model loaded for inference
Running inference smoke test...
  WT: 0.786
  TC: 0.860
  ET: 0.851
Smoke test passed


In [7]:
# ── CELL 7: EVALUATION — All 5 Scenarios ──────────────────────────────────────

scenarios = {
    'clean':         lambda v: v,
    'missing_T1CE':  lambda v: np.concatenate([v[:2], np.zeros_like(v[2:3]), v[3:]], axis=0),
    'missing_FLAIR': lambda v: np.concatenate([np.zeros_like(v[:1]), v[1:]], axis=0),
    'corrupt_T1CE':  lambda v: corrupt_vol(v, 2),
    'corrupt_FLAIR': lambda v: corrupt_vol(v, 0),
}

def corrupt_vol(vols, ch_idx, sigma=0.5):
    out = vols.copy()
    noise = np.random.randn(*out[ch_idx].shape).astype(np.float32) * sigma
    out[ch_idx] = (out[ch_idx] + noise)
    return out

eval_rows = []
print('='*80); print('MedBIND3D v16 — 3D U-Net with Modality Dropout'); print('='*80)

for pd_ in tqdm(test_dirs, desc='Test patients'):
    pid = pd_.name
    try:
        vols, seg, _ = load_patient(pd_)
        gtr = gt_regions(seg)
        row = {'Patient': pid}
        for scenario_name, transform_fn in scenarios.items():
            vols_s = transform_fn(vols)
            preds  = sliding_window_inference(model, vols_s)
            for r in REGIONS:
                row[f'{r}_{scenario_name}'] = dice_3d(preds[r], gtr[r])
            gc.collect(); torch.cuda.empty_cache()
        eval_rows.append(row)
        pd.DataFrame(eval_rows).to_csv(f'{OUTPUT_DIR}/v16_results.csv', index=False)
        print(f'  {pid}: clean ET={row["ET_clean"]:.3f}  miss_T1CE ET={row["ET_missing_T1CE"]:.3f}')
    except Exception as e:
        print(f'  [FAILED] {pid}: {e}'); traceback.print_exc()

df = pd.DataFrame(eval_rows)
print('\n'+'='*80); print('RESULTS'); print('='*80)
for scenario_name in scenarios.keys():
    print(f'\n--- {scenario_name.upper()} ---')
    for r in REGIONS:
        col = f'{r}_{scenario_name}'
        if col in df.columns:
            m=df[col].mean(); s=df[col].std()
            clean_col = f'{r}_clean'
            drop = df[clean_col].mean()-m if scenario_name!='clean' and clean_col in df.columns else 0
            print(f'  {r}: {m:.4f}+/-{s:.4f}  (drop from clean: {drop:+.4f})')

MedBIND3D v16 — 3D U-Net with Modality Dropout


Test patients:   5%|▌         | 1/20 [00:21<06:41, 21.11s/it]

  BraTS20_Training_001: clean ET=0.851  miss_T1CE ET=0.077


Test patients:  10%|█         | 2/20 [00:41<06:14, 20.82s/it]

  BraTS20_Training_002: clean ET=0.586  miss_T1CE ET=0.281


...

  BraTS20_Training_018: clean ET=0.484  miss_T1CE ET=0.106


Test patients:  95%|█████████▌| 19/20 [06:31<00:20, 20.58s/it]

  BraTS20_Training_019: clean ET=0.655  miss_T1CE ET=0.509


Test patients: 100%|██████████| 20/20 [06:52<00:00, 20.61s/it]

  BraTS20_Training_020: clean ET=0.518  miss_T1CE ET=0.297

RESULTS

--- CLEAN ---
  WT: 0.6983+/-0.1485  (drop from clean: +0.0000)
  TC: 0.7303+/-0.1775  (drop from clean: +0.0000)
  ET: 0.6805+/-0.1318  (drop from clean: +0.0000)

--- MISSING_T1CE ---
  WT: 0.6772+/-0.1627  (drop from clean: +0.0211)
  TC: 0.4418+/-0.1823  (drop from clean: +0.2884)
  ET: 0.2974+/-0.1499  (drop from clean: +0.3831)

--- MISSING_FLAIR ---
  WT: 0.4241+/-0.2162  (drop from clean: +0.2742)
  TC: 0.6183+/-0.2857  (drop from clean: +0.1119)
  ET: 0.5526+/-0.2944  (drop from clean: +0.1280)

--- CORRUPT_T1CE ---
  WT: 0.6984+/-0.1496  (drop from clean: -0.0001)
  TC: 0.7299+/-0.1777  (drop from clean: +0.0003)
  ET: 0.6815+/-0.1290  (drop from clean: -0.0010)

--- CORRUPT_FLAIR ---
  WT: 0.6919+/-0.1454  (drop from clean: +0.0063)
  TC: 0.7290+/-0.1772  (drop from clean: +0.0013)
  ET: 0.6802+/-0.1320  (drop from clean: +0.0003)


In [8]:
# ── CELL 8: VISUALIZATIONS (400 DPI) ─────────────────────────────────────────
df = pd.read_csv(f'{OUTPUT_DIR}/v16_results.csv')
matplotlib.rc('font', family='serif', size=9)
CR = ['#3A7D44','#E8871E','#8B5E83']
scenario_labels = ['Clean','Miss T1CE','Miss FLAIR','Corr T1CE','Corr FLAIR']
scenario_cols   = ['clean','missing_T1CE','missing_FLAIR','corrupt_T1CE','corrupt_FLAIR']

fig, axes = plt.subplots(1,3,figsize=(12,3.5))
for ax,r,c in zip(axes,REGIONS,CR):
    means = [df[f'{r}_{s}'].mean() for s in scenario_cols]
    stds  = [df[f'{r}_{s}'].std() for s in scenario_cols]
    ax.bar(scenario_labels, means, yerr=stds, capsize=3, color=c, alpha=0.85,
           edgecolor='black', linewidth=0.5)
    ax.axhline(means[0], color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_title(f'{r}'); ax.set_ylabel('Dice'); ax.set_ylim(0,1.0)
    ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(20); lbl.set_ha('right')

plt.suptitle('v16: 3D U-Net with modality dropout — all scenarios', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v16_fig_results.png', dpi=400, bbox_inches='tight')
plt.close()

# SOTA comparison table
print('='*80)
print('COMPARISON vs RFNet (published BraTS2020 numbers, averaged 15 scenarios)')
print('='*80)
print(f'{"Method":<30}{"WT":<12}{"TC":<12}ET')
print('-'*60)
print(f'{"RFNet (ICCV 2021)":<30}{"86.98":<12}{"78.23":<12}{"61.47"}')
for scenario in ['clean','missing_T1CE']:
    label = f'v16 ({scenario})'
    wt=df[f'WT_{scenario}'].mean(); tc=df[f'TC_{scenario}'].mean(); et=df[f'ET_{scenario}'].mean()
    print(f'{label:<30}{wt*100:<12.2f}{tc*100:<12.2f}{et*100:.2f}')
print(f'\nAll outputs: {os.path.abspath(OUTPUT_DIR)}')

COMPARISON vs RFNet (published BraTS2020 numbers, averaged 15 scenarios)
Method                        WT          TC          ET
------------------------------------------------------------
RFNet (ICCV 2021)             86.98       78.23       61.47
v16 (clean)                   69.83       73.03       68.05
v16 (missing_T1CE)            67.72       44.18       29.74

All outputs: C:\Users\arnav\Desktop\MedBIND3D\MedBIND3D\medclipsam\MedCLIP-SAMv2\medbind3d_v16_outputs
